# Shard creation

In [22]:
# pip install qdrant-client python-dotenv

"""
Qdrant collection + custom shard-key provisioning script.

Key fixes vs. original:
  - list_shard_keys response correctly unwrapped  (ShardKey object → value)
  - Existing shard keys fetched before create, so skip is reliable
  - create_shard_key called with explicit shards_number for clarity
  - "already exists" detection uses a pre-flight set instead of brittle string match
  - TCP probe result actually gates the client creation with a clear error
  - Duplicate QdrantClient import removed
  - Debug log helper kept but isolated; easy to strip
  - Verification section reuses the already-imported client
"""

import json
import os
import socket
import time
from pathlib import Path
from typing import Iterable

from dotenv import load_dotenv
from qdrant_client import QdrantClient, models

# ---------------------------------------------------------------------------
# Debug logging (session-scoped; remove once stable)
# ---------------------------------------------------------------------------

DEBUG_LOG_PATH = "debug-3fed21.log"


def _debug_log(hypothesis_id: str, location: str, message: str, data: dict) -> None:
    payload = {
        "sessionId": "3fed21",
        "runId": "repro",
        "hypothesisId": hypothesis_id,
        "location": location,
        "message": message,
        "data": data,
        "timestamp": int(time.time() * 1000),
    }
    with open(DEBUG_LOG_PATH, "a", encoding="utf-8") as f:
        f.write(json.dumps(payload, ensure_ascii=False) + "\n")


# ---------------------------------------------------------------------------
# Environment
# ---------------------------------------------------------------------------

def load_env() -> Path | None:
    """Walk up from CWD until a .env is found (notebook-friendly)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        env_path = p / ".env"
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


loaded_env = load_env()
_debug_log(
    "H1",
    "create_collections.py:init",
    "Environment discovery",
    {
        "cwd": str(Path.cwd()),
        "loaded_env": str(loaded_env) if loaded_env else None,
        "has_QDRANT_HOST": bool(os.getenv("QDRANT_HOST")),
        "has_QDRANT_PORT": bool(os.getenv("QDRANT_PORT")),
    },
)

host = os.getenv("QDRANT_HOST", "localhost")
port = int(os.getenv("QDRANT_PORT", "6333"))
vector_size = int(os.getenv("QDRANT_VECTOR_SIZE", "1536"))
url = f"http://{host}:{port}"

_debug_log("H2", "create_collections.py:init", "Resolved Qdrant URL",
           {"host": host, "port": port, "url": url})

# ---------------------------------------------------------------------------
# TCP connectivity probe
# ---------------------------------------------------------------------------

try:
    with socket.create_connection((host, port), timeout=2):
        tcp_ok = True
    _debug_log("H3", "create_collections.py:init", "TCP connect ok",
               {"host": host, "port": port})
except Exception as exc:
    tcp_ok = False
    _debug_log("H3", "create_collections.py:init", "TCP connect failed",
               {"host": host, "port": port, "error": repr(exc)})
    raise RuntimeError(
        f"Cannot reach Qdrant at {host}:{port} — check that the server is running "
        f"and QDRANT_HOST / QDRANT_PORT are set correctly.\n  Original error: {exc}"
    ) from exc

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

# In CUSTOM sharding, shard_number = physical shards per shard_key.
shards_per_tenant = int(os.getenv("QDRANT_SHARDS_PER_TENANT", "1"))
replication_factor = int(os.getenv("QDRANT_REPLICATION_FACTOR", "1"))

force_recreate = os.getenv(
    "QDRANT_FORCE_RECREATE_COLLECTIONS", "false"
).strip().lower() in ("1", "true", "yes")

# ---------------------------------------------------------------------------
# Client (single instance reused everywhere)
# ---------------------------------------------------------------------------

client = QdrantClient(url=url)

# ---------------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------------

# Option A: single org
org_name: str = ""
tenant_names: list[str] = []

# Option B: multiple orgs (takes precedence when non-empty)
org_to_tenants: dict[str, list[str]] = {
    "diy-c4ec-prod": [
        "SOCIETAL_THINKING",
        "VJ_PRIVATE",
    ],
    "diy-dasra-prod": ["DASRA"],
    "diy-dasra-prod-v2": ["DASRA"],
    "diy-fishforever-prod": ["FISHFOREVER"],
    "diy-idr-prod": ["IDR_NEW"],
    "diy-mad-prod": ["MAD"],
    "diy-newlongevity-prod": ["NEWLONGEVITY"],
    "diy-pcw-prod": [
        "PCW",
        "PRIMEMEGHALAYA",
        "IMPACT-FAILURE",
        "SUSTAIN_PLUS",
        "ODISHA",
        "IMPACT",
        "KRISHIMITRA",
        "PWD",
        "TORCHBEARERS",
        "PUBLICHEALTH",
        "SAURAEMITRA",
        "HEALTHSTUDY",
        "PHIA",
        "STAGE",
    ],
    "diy-selco-prod": [
        "SELCO_MILLET",
        "LEMELSON",
        "IRENA",
        "SELCO_INTERNAL",
    ],
    "diy-stage-prod": ["STAGE"],
    "diy-socialinnovation-prod": ["SOCIAL_INNOVATION"],
    "diy-gramvani-prod": ["GRAMVANI"],
    "diy-agri-prod": ["AGRI-MUSEUM"],
    "diy-cysd-prod": ["CYSD"],
    "diy-urmul-prod": ["URMUL"],
    "diy-apurva-prod": [
        "APURVA_PUBLIC",
        "AXUM",
        "Selco_Impact_Failure",
        "Public",
        "SELCO_IMPACT_FAILURE",
        "APURVA",
        "MAKEADIFF",
        "SOCIETAL_THINKING",
        "NEWLONGEVITY",
        "FISHFOREVER",
        "DASRA",
        "THEWELLBEINGPROJECT",
        "FES",
        "kuza",
        "RARE",
        "TNKOBOTOOL",
        "SELCO",
        "GRAMVAANI",
        "GOONJ",
        "SERENAS",
        "HAQDARSHAK",
        "AXIS",
        "RANA",
        "CWSINDIA_TRIAL",
        "IFQM",
        "MUSE",
        "SULINTELIGENCIA",
        "AGBA",
    ],
}

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------


def _dedupe_str(values: Iterable[object]) -> list[str]:
    """Deduplicate while preserving insertion order."""
    seen: set[str] = set()
    out: list[str] = []
    for v in values:
        s = str(v)
        if s not in seen:
            seen.add(s)
            out.append(s)
    return out


def _sharding_method(collection_name: str) -> str | None:
    """Return 'custom', 'auto', or None for an existing collection."""
    info = client.get_collection(collection_name=collection_name)
    params = getattr(getattr(info, "config", None), "params", None)
    if params is None:
        return None
    method = getattr(params, "sharding_method", None)
    return str(method).lower() if method is not None else None


def _existing_shard_keys(collection_name: str) -> set[str]:
    """
    Return the set of shard key *values* already registered on a collection.

    FIX: The qdrant-client returns a ShardKeysResponse whose .shard_keys is a
    list of ShardKey objects.  Each ShardKey has a .shard_key attribute that
    holds the actual int or str value.  The original code stopped one level
    short and compared ShardKey objects instead of their inner values.
    """
    try:
        resp = client.list_shard_keys(collection_name=collection_name)
    except Exception:
        return set()

    if not resp or not resp.shard_keys:
        return set()

    result: set[str] = set()
    for entry in resp.shard_keys:
        # entry  → ShardKey
        # entry.shard_key → the actual int | str value
        raw = getattr(entry, "shard_key", None)
        if raw is not None:
            result.add(str(raw))
        else:
            # Fallback: repr the whole object so nothing is silently swallowed
            result.add(str(entry))
    return result


# ---------------------------------------------------------------------------
# Core provisioning
# ---------------------------------------------------------------------------


def ensure_collection_and_tenants(collection_name: str, tenants: list[str]) -> dict:
    """
    Idempotently create a collection and register custom shard keys (tenants).

    - If the collection already exists with the wrong sharding mode and
      force_recreate=True, it is dropped and re-created.
    - Shard keys are fetched first; only missing ones are created.
    """
    tenants = _dedupe_str(tenants)
    want_custom = len(tenants) > 0

    exists = client.collection_exists(collection_name=collection_name)

    if exists and want_custom:
        current = _sharding_method(collection_name)
        is_custom = current is not None and "custom" in current
        if not is_custom:
            if not force_recreate:
                raise RuntimeError(
                    f"Collection '{collection_name}' exists but uses '{current}' sharding "
                    f"(need 'custom'). Set QDRANT_FORCE_RECREATE_COLLECTIONS=true to "
                    f"drop and re-create it."
                )
            print(f"  ⚠  Dropping '{collection_name}' for re-creation (force_recreate=True)")
            client.delete_collection(collection_name=collection_name)
            exists = False

    if not exists:
        kwargs: dict = dict(
            collection_name=collection_name,
            vectors_config=models.VectorParams(
                size=vector_size,
                distance=models.Distance.COSINE,
            ),
            replication_factor=replication_factor,
        )
        if want_custom:
            kwargs["shard_number"] = shards_per_tenant
            kwargs["sharding_method"] = models.ShardingMethod.CUSTOM
        client.create_collection(**kwargs)
        print(f"  ✓ Created collection '{collection_name}'" +
              (" (custom sharding)" if want_custom else ""))

    # -----------------------------------------------------------------------
    # Shard key creation  —  pre-flight fetch avoids brittle string matching
    # -----------------------------------------------------------------------
    created_keys: list[str] = []
    skipped_keys: list[str] = []

    if want_custom:
        already_present = _existing_shard_keys(collection_name)

        for shard_key in tenants:
            if str(shard_key) in already_present:
                skipped_keys.append(shard_key)
                continue

            try:
                client.create_shard_key(
                    collection_name=collection_name,
                    shard_key=shard_key,
                    shards_number=shards_per_tenant,
                )
                created_keys.append(shard_key)
            except Exception as exc:
                msg = str(exc).lower()
                if "distributed mode disabled" in msg or (
                    "cluster" in msg and "disabled" in msg
                ):
                    raise RuntimeError(
                        "Qdrant cluster/distributed mode is disabled — shard keys cannot "
                        "be created. Enable it in your Qdrant config (see deploy/qdrant/), "
                        "then re-run."
                    ) from exc
                # Race condition: another process created the key between our
                # list and our create — treat as success.
                if "already exists" in msg or "conflict" in msg:
                    skipped_keys.append(shard_key)
                else:
                    raise

    return {
        "collection": collection_name,
        "tenants": tenants,
        "custom_sharding": want_custom,
        "created_shard_keys": created_keys,
        "skipped_shard_keys": skipped_keys,
    }


# ---------------------------------------------------------------------------
# Build workset and run
# ---------------------------------------------------------------------------

if org_to_tenants:
    workset = {k: _dedupe_str(v) for k, v in org_to_tenants.items()}
elif org_name:
    workset = {org_name: _dedupe_str(tenant_names)}
else:
    workset = {}

if not workset:
    print("Set org_name + tenant_names  OR  org_to_tenants at the top of this file, then re-run.")
else:
    print(f"Qdrant: {url}")
    print(f"Vector size: {vector_size}  |  distance: Cosine")
    print(f"Shards per tenant: {shards_per_tenant}  |  replication: {replication_factor}")
    print()

    results = []
    for org, tenants in workset.items():
        print(f"▶ {org}  ({len(tenants)} tenants)")
        result = ensure_collection_and_tenants(org, tenants)
        results.append(result)
        print(f"    created: {result['created_shard_keys']}")
        print(f"    skipped: {result['skipped_shard_keys']}")

    print(f"\nDone. Processed {len(results)} collections.")

# ---------------------------------------------------------------------------
# Verification  —  correctly unwraps ShardKey objects
# ---------------------------------------------------------------------------

print("\n--- Verification ---")

if not workset:
    print("Nothing to verify.")
else:
    for org in workset:
        try:
            keys = _existing_shard_keys(org)   # reuses the fixed helper
            print(f"{org}: {len(keys)} shard_key(s) → {sorted(keys)}")
        except Exception as exc:
            print(f"{org}: ERROR — {exc}")

Qdrant: http://localhost:6333
Vector size: 1536  |  distance: Cosine
Shards per tenant: 1  |  replication: 1

▶ diy-c4ec-prod  (2 tenants)
    created: []
    skipped: ['SOCIETAL_THINKING', 'VJ_PRIVATE']
▶ diy-dasra-prod  (1 tenants)
    created: []
    skipped: ['DASRA']
▶ diy-dasra-prod-v2  (1 tenants)
    created: []
    skipped: ['DASRA']
▶ diy-fishforever-prod  (1 tenants)
    created: []
    skipped: ['FISHFOREVER']
▶ diy-idr-prod  (1 tenants)
    created: []
    skipped: ['IDR_NEW']
▶ diy-mad-prod  (1 tenants)
    created: []
    skipped: ['MAD']
▶ diy-newlongevity-prod  (1 tenants)
    created: []
    skipped: ['NEWLONGEVITY']
▶ diy-pcw-prod  (14 tenants)
    created: []
    skipped: ['PCW', 'PRIMEMEGHALAYA', 'IMPACT-FAILURE', 'SUSTAIN_PLUS', 'ODISHA', 'IMPACT', 'KRISHIMITRA', 'PWD', 'TORCHBEARERS', 'PUBLICHEALTH', 'SAURAEMITRA', 'HEALTHSTUDY', 'PHIA', 'STAGE']
▶ diy-selco-prod  (4 tenants)
    created: []
    skipped: ['SELCO_MILLET', 'LEMELSON', 'IRENA', 'SELCO_INTERNAL']
▶ d

# Shards created Verification

In [ ]:
for org in workset:
    keys = _existing_shard_keys(org)
    print(f"{org}: {sorted(keys)}")

diy-c4ec-prod: ["key='SOCIETAL_THINKING'", "key='VJ_PRIVATE'"]
diy-dasra-prod: ["key='DASRA'"]
diy-dasra-prod-v2: ["key='DASRA'"]
diy-fishforever-prod: ["key='FISHFOREVER'"]
diy-idr-prod: ["key='IDR_NEW'"]
diy-mad-prod: ["key='MAD'"]
diy-newlongevity-prod: ["key='NEWLONGEVITY'"]
diy-pcw-prod: ["key='HEALTHSTUDY'", "key='IMPACT'", "key='IMPACT-FAILURE'", "key='KRISHIMITRA'", "key='ODISHA'", "key='PCW'", "key='PHIA'", "key='PRIMEMEGHALAYA'", "key='PUBLICHEALTH'", "key='PWD'", "key='SAURAEMITRA'", "key='STAGE'", "key='SUSTAIN_PLUS'", "key='TORCHBEARERS'"]
diy-selco-prod: ["key='IRENA'", "key='LEMELSON'", "key='SELCO_INTERNAL'", "key='SELCO_MILLET'"]
diy-stage-prod: ["key='STAGE'"]
diy-socialinnovation-prod: ["key='SOCIAL_INNOVATION'"]
diy-gramvani-prod: ["key='GRAMVANI'"]
diy-agri-prod: ["key='AGRI-MUSEUM'"]
diy-cysd-prod: ["key='CYSD'"]
diy-urmul-prod: ["key='URMUL'"]
diy-apurva-prod: ["key='AGBA'", "key='APURVA'", "key='APURVA_PUBLIC'", "key='AXIS'", "key='AXUM'", "key='CWSINDIA_TRIAL'"

: 